<div style="display: flex; justify-content: space-between; align-items: flex-start; border-bottom: 2px solid #555555; padding-bottom: 15px; margin-bottom: -8px;">
    <div style="width: 50%;">
        <h2>
            <span style="color: #B30033;">▍</span>TRABAJO DE FIN DE MÁSTER
        </h2>
        <h1 style="margin-top: -10px;">MARCO DE TRABAJO DE RECOMENDACIÓN DE PERSONAL PARA LA FORMACIÓN DE EQUIPOS EN PROYECTOS DE INGENIERÍA DEL SOFTWARE</h1>
    </div>
    <div style="width: 50%; text-align: right;">
        <div style="display: flex; justify-content: space-between; align-items: flex-start; margin-top: 30px;">
            <div style="width: 20%;"></div>
            <div style="width: 80%; border-left: 2px solid #555555; padding-left: 20px;">
                <div style="margin-bottom: 20px;">
                    <p style="margin: 0; font-size: 1.4em; font-weight: bold;">
                        TUTORES
                    </p>
                </div>
                <div style="margin-top: 8px; text-align: right;">
                    <span style="display: block; font-size: 1em; color: #D0D0D0;">
                        Aurora Vizcaíno Barcelo
                    </span>
                    <span style="display: block; font-size: 1em; color: #D0D0D0; margin-top: 5px;">
                        Julio Alberto López Gómez
                    </span>
                </div>
            </div>
        </div>
    </div>
</div>

<div style="margin-top: 30px; padding: 20px; background-color: rgba(179, 0, 51, 0.05); border-left: 4px solid #B30033; border-radius: 4px;">
    <h3 style="margin-top: 0; color: #B30033;">🎯 Objetivo de esta Libreta (Fase 5: Demostración del Framework y Ranking)</h3>
    <p style="font-size: 1.1em; line-height: 1.6; color: #E0E0E0;">
        Esta libreta culmina el desarrollo del framework. Su objetivo es emular un caso de uso real donde, dados los datos de un proyecto entrante y nuestra base de empleados, un <strong>motor de matching explicable</strong> evalúa la idoneidad de los candidatos utilizando métricas de distancia y las reglas de conocimiento extraídas previamente.
    </p>
    <ul style="font-size: 1.1em; line-height: 1.6; color: #D0D0D0; margin-bottom: 0;">
        <li><strong>Ingesta del Proyecto:</strong> Carga de un conjunto de proyectos sintéticos definidos mediante atributos reales (stack tecnológico, políticas de remoto, tamaño de equipo, etc.).</li>
        <li><strong>Ranking de Hard Skills (Competencias Duras):</strong> Evaluación de la idoneidad técnica midiendo la distancia de Manhattan (criterio de ordenación) y la similitud del coseno (métrica complementaria) entre los requisitos del proyecto y el perfil del candidato (datos 2025).</li>
        <li><strong>Ranking de Soft Skills (Competencias Blandas):</strong> Evaluación en dos fases sobre el perfil psicológico y laboral del candidato (datos 2019): primero se descartan los candidatos que activan riesgos mediante reglas expertas ponderadas con la evidencia minada (sobrecarga de trabajo, burnout, aislamiento social, etc.); después, entre los candidatos seguros, se ordena por un índice de compromiso/disponibilidad calculado a partir de sus respuestas de la encuesta (satisfacción laboral, búsqueda activa de empleo, contribución a proyectos de código abierto, etc.).</li>
        <li><strong>Justificación Metodológica:</strong> Se presentan dos rankings separados debido a la naturaleza de las fuentes de datos (años distintos y, posiblemente, candidatos distintos).</li>
    </ul>
</div>
<br/>

<div style="border-bottom: 2px solid #555555; padding-bottom: 25px; margin-bottom: 10px">
    <div style="display: flex; align-items: center; margin-bottom: 10px;">
        <span style="color: #B30033; font-size: 1.5em; margin-right: 10px;">▍</span>
        <h3 style="margin: 0; font-size: 1.4em; font-weight: bold">
            Realizado por Miguel Ángel Sánchez-Medina Camacho <span style="font-size: 0.7em; font-style: italic; font-weight: normal;"></span>
        </h3>
    </div>
</div>

### 5.1. Configuración y Carga de Datos

Cargamos las librerías necesarias, las reglas expertas, el json con los proyectos sintéticos, las matrices de candidatos de ambos pipelines (hard skills 2025 y soft skills 2019) y el mapa de países. Los proyectos los importamos aquí todos al mismo tiempo, y la elección de para cuáles queremos buscar candidatos se hará más adelante, en las secciones 5.2 y 5.4.

Cabe destacar que la mayor parte de la lógica del modelo (cálculo de distancias, activación de reglas de riesgo, cálculo del índice de compromiso, etc.) no se incorpora en esta libreta, sino que se implementó en dos módulos auxiliares que importamos aquí mismo, `modelo_aux.py` para hard skills y `modelo_aux_soft.py` para soft skills. Aquí en la libreta solo quedan las llamadas y la presentación de resultados, hecho así para que sea más fácil de mantener y de seguir el flujo y objetivos de la libreta de una forma más ordenada.

In [1]:
import os
import sys
import json
import pandas as pd
import numpy as np

sys.path.append(os.path.abspath(os.path.join('../..')))
from src.hard_skills import config as config_hard_skills
from src.soft_skills import config as config_soft_skills
from src.hard_skills import modelo_aux
from src.soft_skills import modelo_aux_soft

with open(config_hard_skills.REGLAS_EXPERTAS_PATH, encoding='utf-8') as f:
    reglas_expertas = json.load(f)['reglas']

with open(config_hard_skills.PROYECTOS_PATH, encoding='utf-8') as f:
    proyectos = json.load(f)['proyectos']

reglas_soft_proyecto = [r for r in reglas_expertas if r['tipo'] == 'soft_parcial']
reglas_hard_proyecto = [r for r in reglas_expertas if r['tipo'] == 'hard_parcial']


df_candidatos_hard_skills = pd.read_csv(config_hard_skills.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(int)
df_candidatos_soft_skills = pd.read_csv(config_soft_skills.MATRIZ_TRANSACCIONAL_PATH, index_col='CandidateId').fillna(0).astype(int)

paises_mapa = pd.read_csv(config_hard_skills.PAISES_MAPA_PATH, sep=';')


print(f"Candidatos hard skills (2025): {len(df_candidatos_hard_skills)}")
print(f"Candidatos soft skills (2019): {len(df_candidatos_soft_skills)}")
print(f"Proyectos sintéticos cargados: {len(proyectos)}")
print(f"Países cargados: {len(paises_mapa)} ")
print(f"Reglas soft_parcial evaluables con proyecto real: {len(reglas_soft_proyecto)}")
print(f"Reglas hard_parcial evaluables con proyecto real: {len(reglas_hard_proyecto)}")

Candidatos hard skills (2025): 35434
Candidatos soft skills (2019): 88778
Proyectos sintéticos cargados: 20
Países cargados: 169 
Reglas soft_parcial evaluables con proyecto real: 9
Reglas hard_parcial evaluables con proyecto real: 1


Ahora enriquecemos los datos de los candidatos de 2025 (hard skills) con información geográfica: país, continente e idiomas oficiales de los diferentes candidatos presentes en los datos. Más adelante esta información es imprescindible para comprobar la localización y el idioma de los candidatos con los requisitos del proyecto.

In [2]:
df_geo_hard = pd.read_csv(config_hard_skills.PROCESSED_EDA_PATH, index_col='CandidateId', usecols=['CandidateId', 'Country', 'Continente'])
df_geo_hard = df_geo_hard.reset_index().merge(
    paises_mapa[['Pais_Original', 'Idiomas_Oficiales']],
    left_on='Country', right_on='Pais_Original', how='left'
).set_index('CandidateId')

### 5.2. Ranking de Hard Skills — distancia Manhattan y similitud Coseno

Para cada proyecto, se compara a los candidatos con un vector ideal construido a partir de sus requisitos (p.ej. tecnologías, seniority o nivel de experiencia, % de teletrabajo, idioma, etc.) usando dos métricas ponderadas: la distancia de Manhattan, la cual mide el desajuste absoluto en cada dimensión, y la similitud del coseno, que tiene como objetivo medir la orientación general del perfil de los candidatos con respecto al perfil ideal. El ranking en este caso se ordena por la distancia de Manhattan únicamente, por lo que la similitud coseno la mostramos como una métrica complementaria, solo para aportar más datos pero no como criterio para ordenar.

La construcción de este vector candidato/vector ideal y el cálculo de los pesos de cada dimensión (`vector_candidato_proyecto`, `obtener_pesos`) se han añadido en `modelo_aux.py`. Allí también se aplica la fórmula de la distancia y la similitud como tal (`asignacion_proyecto_candidatos_hard`).

In [3]:
def asignacion_proyecto_candidatos_hard(proyecto, df_matriz, top_n=5):
    return modelo_aux.asignacion_proyecto_candidatos_hard(proyecto, df_matriz, df_geo_hard, paises_mapa, config_hard_skills, top_n=top_n)

Aparte de la distancia Manhattan, se incorporan evidencias estadísticas, por lo que en esta celda vamos a cargar las reglas de asociación minadas con Apriori (libreta `03_MODEL_TRAINING_HARD_SKILLS`) sobre las combinaciones de parámetros de los candidatos, y después construimos `items_totales` (esta es la unión de antecedente y consecuente de cada regla) con el fin de poder comprobar más adelante si las tecnologías del proyecto que domina cada candidato coinciden con alguna combinación que normalmente aparece junta con frecuencia. Esto es lo que usa `anadir_evidencia_mineria` para complementar el ranking con un lift real, como respaldo estadístico adicional a la distancia al perfil ideal (el orden del ranking sigue dependiendo solo de la distancia Manhattan).

In [4]:
reglas_limpias_hard = pd.read_csv(config_hard_skills.REGLAS_LIMPIAS_PATH)
reglas_limpias_hard['items_totales'] = reglas_limpias_hard.apply(
    lambda row: set(str(row['antecedents']).split(' | ')) | set(str(row['consequents']).split(' | ')),
    axis=1
)
print(f"Reglas de minería cargadas: {len(reglas_limpias_hard)}")

Reglas de minería cargadas: 10107


La función `modelo_aux.add_evidencia` requiere de argumentos como el ranking que queremos completar, el proyecto objetivo, la matriz de candidatos, las reglas minadas y la configuración de las hard skills. De estos, las reglas minadas (`reglas_limpias_hard`) y la configuración (`config_hard_skills`) son siempre las mismas en toda la libreta. Por ello, definimos aquí `anadir_evidencia_mineria`, la cual únicamente nos pide lo que sí cambia, como el ranking, el proyecto y la matriz de candidatos, y ya se encarga por dentro de pasarle esos dos valores fijos a `add_evidencia`. De esta forma, cada vez que necesitemos añadir evidencia a un ranking, escribimos menos y evitamos repetir los mismos dos argumentos una y otra vez.

In [5]:
def anadir_evidencia_mineria(resultado_ranking, proyecto, df_matriz, top_n_evidencia=4, max_items_evidencia=4):
    return modelo_aux.add_evidencia(resultado_ranking, proyecto, df_matriz, reglas_limpias_hard, config_hard_skills,
        top_n_evidencia=top_n_evidencia, max_items_evidencia=max_items_evidencia)

Ahora para esta fase, vamos a hacer una pequeña prueba con el primer proyecto del dataset (`proyectos[0]`, PRY-2026-001): mostramos la distancia Manhattan y la similitud del coseno, añadimos la evidencia y mostramos el resultado con `modelo_aux.mostrar_ranking_proyecto`.

Esta celda nos da como salida el ranking final de las `hard skills` dividido en dos bloques. El primero resume de qué forma se interpreta el proyecto: la oferta laboral, empresa, rol, ID, industria, requisitos técnicos requeridos, nivel de experiencia objetivo, % teletrabajo, idioma, países involucrados y las herramientas de trabajo. Debajo de esta lista vemos los campos del proyecto que no se pueden evaluar porque falta el dato o el mapeo (por ejemplo, un SSOO que el proyecto no define y deja a elección de los candidatos, o una herramienta como Slack que no tiene equivalente en la encuesta). Estos campos se quedan fuera de la comparación y no perjudican ni benefician a ninguno de los profesionales a evaluar.

Para el segundo bloque, tenemos el top 5 candidatos, ordenados de mayor a menor nivel de match en base a la distancia Manhattan (de menor a mayor). Para cada uno se muestran su ubicación, idiomas, qué requisitos del proyecto cumple y cuáles no, y si existe evidencia de la minería o lift que respalde la combinación de tecnologías (cuando ninguna regla minada la reproduce, decimos "sin evidencia minada").

In [6]:
resultado_prueba = asignacion_proyecto_candidatos_hard(proyectos[0], df_candidatos_hard_skills)
resultado_prueba = anadir_evidencia_mineria(resultado_prueba, proyectos[0], df_candidatos_hard_skills)
modelo_aux.mostrar_ranking_proyecto(proyectos[0], resultado_prueba, df_candidatos_hard_skills, df_geo_hard, paises_mapa, config_hard_skills)

 OFERTA: NovaPay FinTech busca Data Engineer / Backend
 PROYECTO: PRY-2026-001 | Industria: FinTech
 REQUISITOS TÉCNICOS: Python, SQL, AWS
 SENIORITY: Senior (umbral >= 0.7)
 TELETRABAJO: 40% | IDIOMA: Spanish
 PAÍSES DEL PROYECTO: Spain, Portugal
 HERRAMIENTAS: Slack, Jira, GitHub
 (No evaluables por falta de mapeo/dato: os: no especificado; tools_used:Slack; ai_tools_preferred: no especificado)

TOP 5 CANDIDATOS:

 RANKING #1 | Candidato 33137 | Nivel de Match: 96.32% (distancia Manhattan: 0.98 | similitud coseno: 98.78%)
    * Ubicación: Spain | Idiomas: Spanish
    * Cumple (15/16): Python, SQL, Amazon Web Services (AWS), Nivel de seniority, % Teletrabajo, Industria, Rol / puesto, Idioma del proyecto, Localización, Nivel de gestión, Formación, Tipo de contrato, Herramientas de trabajo, Presupuesto salarial, Tamaño de empresa
    * No cumple: IDE preferido
    * Evidencia de minería (lift, sobre 3 ítems tech): sin evidencia minada
----------------------------------------------------

### 5.3. Ranking de Soft Skills — riesgo activado por las reglas `soft_parcial`

Para esta sección vamos a construir el motor de riesgo psicosocial. Para ello disponemos de 9 reglas expertas `soft_parcial` que tratan temas como la sobrecarga de trabajo, burnout, aislamiento social, riesgo de conflictos, resistencia al cambio, fatiga digital y falta de soporte al trabajo remoto. 7 de esas 9 reglas se encuentran reforzadas con una evidencia estadística real como es el lift, que se calcula por minería de reglas de asociación cuando existe, o de forma manual cuando el ítem no aparece en las reglas minadas; las otras 2 (nº 11 y 22) dependen de un único ítem sin evidencia posible y reciben un peso fijo.

Aquí se evalúa el riesgo a nivel de toda la población de candidatos (para el dataset de 2019) sin referencia todavía a ningún proyecto, por lo que es el punto del que partimos para ver qué candidatos, en general, son más o menos propensos a activar riesgos. La aplicación de las reglas a un proyecto real la veremos más adelante, en la sección 5.4.

In [7]:
reglas_limpias_soft = pd.read_csv(config_soft_skills.REGLAS_LIMPIAS_PATH)
reglas_limpias_soft['items_totales'] = reglas_limpias_soft.apply(
    lambda row: set(str(row['antecedents']).split(' | ')) | set(str(row['consequents']).split(' | ')),
    axis=1
)
print(f"Reglas de minería (soft) cargadas: {len(reglas_limpias_soft)}")

Reglas de minería (soft) cargadas: 9575


Antes de continuar aplicando las reglas de riesgo, vamos a calcular el lift que respalda a cada una de estas 9 reglas `soft_parcial`. Para cada una hay 3 posibilidades: en caso de que exista alguna regla minada con Apriori que contenga los mismos ítems, usamos el mayor lift entre ellas (fuente `apriori`); si la regla combina exactamente 2 ítems y no hay evidencia minada, lo recalculamos sobre la matriz de candidatos (fuente `calculo_manual`); y en caso de que la regla dependa de un único ítem sin evidencias posibles (como ocurre con las reglas 11 y 22) no hay lift que calcular, y más tarde le asignaremos un peso fijo.

El resultado de esta celda es un diccionario indexado por id de regla, con el nombre del riesgo que tiene asociado y su evidencia calculada. Aquí mostramos estos resultados solo para comprobar que las 9 reglas tienen datos razonables y que coinciden con los valores obtenidos en la libreta 4.

In [8]:
evidencia_reglas_soft = {
    regla['id']: {'riesgo': regla['riesgo'], **modelo_aux_soft.obtener_evidencias_reglas_soft(regla, reglas_limpias_soft, df_candidatos_soft_skills)}
    for regla in reglas_soft_proyecto
}
for id_regla, ev in evidencia_reglas_soft.items():
    print(id_regla, ev)

8 {'riesgo': 'Sobrecarga de trabajo', 'lift': np.float64(2.163680021789325), 'fuente': 'calculo_manual'}
10 {'riesgo': 'Burnout', 'lift': np.float64(1.4195331745659936), 'fuente': 'calculo_manual'}
11 {'riesgo': 'Aislamiento social', 'lift': None, 'fuente': 'sin_evidencia_estadistica'}
12 {'riesgo': 'Riesgo de conflictos', 'lift': np.float64(1.4818591512564103), 'fuente': 'apriori'}
22 {'riesgo': 'Falta de soporte para el trabajo remoto', 'lift': None, 'fuente': 'sin_evidencia_estadistica'}
23 {'riesgo': 'Resistencia al cambio', 'lift': np.float64(1.335074331327406), 'fuente': 'apriori'}
24 {'riesgo': 'Fatiga digital', 'lift': np.float64(1.2397979793224225), 'fuente': 'calculo_manual'}
25 {'riesgo': 'Burnout', 'lift': np.float64(1.451108292637099), 'fuente': 'calculo_manual'}
26 {'riesgo': 'Burnout', 'lift': np.float64(1.5273948178368135), 'fuente': 'apriori'}


Para esta próxima celda vamos a calcular el riesgo para cada candidato de todo el dataset de 2019, frente a las 9 reglas `soft_parcial` a la vez. Este es el escenario más exigente posible, como si todas las reglas aplicasen.

La puntuación `score_riesgo` es la suma de los pesos de evidencia de todas las reglas que activa cada candidato (el lift, o el peso fijo en el caso de las reglas 11 y 22): cuantas más reglas dispare y más fuerte sea su evidencia, mayor el score.

`value_counts().sort_index()` muestra cuántos candidatos comparten cada valor exacto de `score_riesgo`, ordenados de menor a mayor. Esto es un primer vistazo a la distribución de riesgo de toda la población antes de aplicar filtros para un proyecto específico. Más adelante, en la sección 5.4, solo se tendrán en cuenta las reglas que ese proyecto realmente activa en base a sus propios atributos.

In [9]:
df_riesgo_soft = modelo_aux_soft.evaluar_riesgo_poblacion(df_candidatos_soft_skills,reglas_soft_proyecto, evidencia_reglas_soft,config_soft_skills)
df_riesgo_soft['score_riesgo'].value_counts().sort_index()

score_riesgo
0.000000     29305
1.335074      8277
1.419533       574
1.481859     11869
1.527395     10755
             ...  
9.717055         7
9.747806         4
9.863840        14
10.454768        7
11.198915       10
Name: count, Length: 72, dtype: int64

Ahora vamos a mostrar el top 5 candidatos más seguros y expuestos de toda la población, con el detalle de cada uno vía `modelo_aux_soft.riesgo_candidatos`.

En "candidatos más seguros", los 5 salen con score 0.00 y "Sin riesgos activados", lo que nos señala que no hay ningún mérito especial que los distinga entre sí: aquí tenemos 29.305 candidatos con riesgo cero, y al estar todos empatados, el orden de ellos depende de su posición en la tabla, no hay ninguna diferencia real de riesgo.

Para los "candidatos más expuestos" ocurre algo parecido, pero en el extremo contrario: los 5 primeros empatan con 11.2 de score, con exactamente las mismas 8 reglas activas, los mismos ítems y mitigaciones. Esto es coherente con la forma en la que están construidas las reglas: como dependen de variables discretizadas en pocas categorías (por ejemplo, `Neuroticismo_Nivel_Alto` o `WorkWeekHrs_Bin_>55h`), muchos candidatos distintos acaban compartiendo exactamente el mismo perfil de riesgo — no es un error, es un efecto esperado de trabajar con variables categóricas. Además, el máximo son 8 reglas y no 9, porque la nº 8 (`WorkWeekHrs_Bin_>55h`) y la nº 10 (`WorkWeekHrs_Bin_45-55h`) dependen de tramos de horas excluyentes y no pueden activarse a la vez.

Un detalle a destacar es que en la salida muestra "7 riesgos únicos" pero "8 reglas activadas". Esto no es un error, es así porque dos reglas distintas (la nº 25 y la 26) comparten el nombre "Burnout", por causas distintas cada una. Al contar riesgos únicos por nombre, ese "Burnout" repetido solo cuenta una vez; al contar reglas activadas, sí se cuentan las dos por separado.

In [10]:
print("=" * 72)
print(" RANKING DE RIESGO SOFT SKILLS — candidatos más seguros")
print("=" * 72)
for cid in modelo_aux_soft.ranking_riesgos(df_riesgo_soft, ascendente=True, top_n=5).index:
    modelo_aux_soft.riesgo_candidatos(cid, df_riesgo_soft)
    print("-" * 72)

print("\n" + "=" * 72)
print(" RANKING DE RIESGO SOFT SKILLS — candidatos más expuestos")
print("=" * 72)
for cid in modelo_aux_soft.ranking_riesgos(df_riesgo_soft, ascendente=False, top_n=5).index:
    modelo_aux_soft.riesgo_candidatos(cid, df_riesgo_soft)
    print("-" * 72)

 RANKING DE RIESGO SOFT SKILLS — candidatos más seguros
CANDIDATO 1 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
CANDIDATO 2 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
CANDIDATO 8 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
CANDIDATO 9 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
CANDIDATO 11 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------

 RANKING DE RIESGO SOFT SKILLS — candidatos más expuestos
CANDIDATO 17216 | S

### 5.4. Demostración con un proyecto real

Hasta ahora hemos evaluado el riesgo de la población completa con respecto a las 9 reglas a la vez, sin tener en cuenta un proyecto determinado. Ahora, aquí aplicamos el filtro por proyecto, donde cada proyecto activa solo el subconjunto de reglas que le corresponde según sus propios atributos (política de desconexión, solapamiento horario, complejidad, etc.). En primer lugar, vamos a comparar dos casos: `proyectos[0]` (PRY-2026-001), el mismo que usamos en el ranking de hard skills, y `proyectos[3]` (PRY-2026-004), para comprobar que el modelo responde coherentemente según las reglas que activa cada uno. Después, terminamos combinando este filtro con el índice de compromiso para obtener la selección final de candidatos en dos fases, comparando un proyecto de buenas prácticas (`proyectos[11]`) y uno de malas prácticas (`proyectos[1]`).

In [11]:
proyecto_real = proyectos[0]
df_riesgo_proyecto, reglas_activas_proyecto = modelo_aux_soft.evaluacion_riesgos_proyecto(
    proyecto_real, df_candidatos_soft_skills, reglas_soft_proyecto, evidencia_reglas_soft, config_soft_skills
)
top_expuestos_proyecto = modelo_aux_soft.ranking_riesgos(df_riesgo_proyecto, ascendente=False, top_n=5)
modelo_aux_soft.mostrar_riesgo_proyecto(
    proyecto_real, top_expuestos_proyecto, reglas_activas_proyecto,
    total_reglas_soft=len(reglas_soft_proyecto), titulo_bloque="candidatos más expuestos"
)

 PROYECTO: PRY-2026-001 | NovaPay FinTech busca Data Engineer / Backend
 CONDICIONES DE RIESGO QUE ACTIVA ESTE PROYECTO (0/9 reglas soft_parcial):
   Ninguna — las condiciones de este proyecto no activan ningún riesgo de soft skills conocido.

TOP 5 CANDIDATOS MÁS EXPUESTOS PARA ESTE PROYECTO:

RANKING #1
CANDIDATO 1 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
RANKING #2
CANDIDATO 2 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
RANKING #3
CANDIDATO 3 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
------------------------------------------------------------------------
RANKING #4
CANDIDATO 4 | Score de riesgo: 0.00 (0 riesgos únicos, 0 reglas activadas)
  Sin riesgos activados.
-----------------------------------------

In [12]:
proyecto_real = proyectos[3]
df_riesgo_proyecto, reglas_activas_proyecto = modelo_aux_soft.evaluacion_riesgos_proyecto(
    proyecto_real, df_candidatos_soft_skills, reglas_soft_proyecto, evidencia_reglas_soft, config_soft_skills
)
top_expuestos_proyecto = modelo_aux_soft.ranking_riesgos(df_riesgo_proyecto, ascendente=False, top_n=5)
modelo_aux_soft.mostrar_riesgo_proyecto(
    proyecto_real, top_expuestos_proyecto, reglas_activas_proyecto,
    total_reglas_soft=len(reglas_soft_proyecto), titulo_bloque="candidatos más expuestos"
)

 PROYECTO: PRY-2026-004 | ByteForge Studios busca Gameplay Programmer
 CONDICIONES DE RIESGO QUE ACTIVA ESTE PROYECTO (2/9 reglas soft_parcial):
   - [23] Resistencia al cambio: Si el equipo tiene baja apertura a la experiencia y el proyecto es de alta complejidad, podrá surgir resistencia al cambio.
   - [26] Burnout: Si hay un número excesivo de reuniones y múltiples equipos distribuidos, podrá surgir burnout.

TOP 5 CANDIDATOS MÁS EXPUESTOS PARA ESTE PROYECTO:

RANKING #1
CANDIDATO 16 | Score de riesgo: 2.86 (2 riesgos únicos, 2 reglas activadas)
  - [Resistencia al cambio] activado por: Apertura_Nivel_Bajo (peso evidencia: 1.34)
      Mitigación sugerida: Aplicar planes de adopción progresiva, mentoría en áreas nuevas y limitar cambios simultáneos.
  - [Burnout] activado por: WorkChallenge_Meetings (peso evidencia: 1.53)
      Mitigación sugerida: Limitar la duración de las reuniones y priorizar la comunicación asíncrona.
------------------------------------------------------------

Aplicamos ahora el proceso completo con sus dos fases, para un proyecto de buenas prácticas (`proyectos[11]`) y uno de malas prácticas (`proyectos[1]`). `modelo_aux_soft.mejores_candidatos` hace, para cada uno: primero descarta a los candidatos cuyo riesgo (calculado solo con las reglas que ese proyecto concreto activa) supera un umbral máximo — por defecto, riesgo estrictamente cero (`umbral_riesgo_max=0.0`); de los que quedan, los ordena de mayor a menor índice de compromiso/disponibilidad y devuelve el top 10.

En la cabecera de los bloques mostramos cuántas reglas activa ese proyecto en concreto y cuántos candidatos de la población total (88.778) pasan el filtro de riesgo cero. Un proyecto de buenas prácticas, con solo 1 regla activa, deja un pool mucho más amplio (86.282 candidatos) que uno de malas prácticas con las 9 activas (29.305), lo que confirma que las condiciones reales del proyecto cambian de forma sustancial qué candidatos son viables.

En la tabla de salida con el top 10 final, `score_riesgo` y `n_riesgos_unicos` aparecen a 0 en todas las filas. Este resultado no es ninguna sorpresa, porque aquí solo llegan aquellos candidatos que ya pasaron el filtro de riesgo cero en la primera fase. La columna que de verdad diferencia y ordena a estos candidatos es `indice_compromiso`: cuanto más cerca de 1, más disponible o comprometido resulta el candidato según las variables adicionales de la encuesta (satisfacción laboral, búsqueda activa de empleo, aportación a código abierto, etc.).

In [13]:
for etiqueta, proyecto_real in [("proyectos[11] (buenas prácticas)", proyectos[11]),
                                  ("proyectos[1] (malas prácticas)", proyectos[1])]:
    ranking, reglas_activas, n_seguros = modelo_aux_soft.mejores_candidatos(
        proyecto_real, df_candidatos_soft_skills, reglas_soft_proyecto,
        evidencia_reglas_soft, config_soft_skills, top_n=10
    )
    print("=" * 72)
    print(f" {etiqueta} — reglas activas: {len(reglas_activas)} | candidatos con riesgo cero: {n_seguros}")
    print("=" * 72)
    display(ranking[['score_riesgo', 'n_riesgos_unicos', 'indice_compromiso']])
    print()

 proyectos[11] (buenas prácticas) — reglas activas: 1 | candidatos con riesgo cero: 86282


,score_riesgo,n_riesgos_unicos,indice_compromiso
CandidateId,,,
86649,0.0,0,1.000000
5730,0.0,0,1.000000
80059,0.0,0,0.972222
33807,0.0,0,0.972222
41621,0.0,0,0.972222
81122,0.0,0,0.972222
53206,0.0,0,0.972222
62273,0.0,0,0.962222
84812,0.0,0,0.962222



 proyectos[1] (malas prácticas) — reglas activas: 9 | candidatos con riesgo cero: 29305


,score_riesgo,n_riesgos_unicos,indice_compromiso
CandidateId,,,
88115,0.0,0,0.916667
40507,0.0,0,0.916667
20175,0.0,0,0.916667
22340,0.0,0,0.906667
80789,0.0,0,0.906667
54565,0.0,0,0.906667
35142,0.0,0,0.888889
20843,0.0,0,0.888889
20926,0.0,0,0.888889
